In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lexicalrichness import LexicalRichness
from scipy.stats import pearsonr, spearmanr
from matplotlib.patches import Patch


# ============================================================
# 1. LEXICAL FEATURE FUNCTIONS
# ============================================================

def word_count(text):
    lex = LexicalRichness(text)
    return lex.words

def term_count(text):
    lex = LexicalRichness(text)
    return lex.terms

def ttr(text):
    lex = LexicalRichness(text)
    return lex.ttr

def mtld(text, threshold=0.72):
    lex = LexicalRichness(text)
    return lex.mtld(threshold=threshold)

def hdd(text, draws=42):
    lex = LexicalRichness(text)
    return lex.hdd(draws=draws)

def vocd(text, ntokens=50, within_sample=100, iterations=3):
    lex = LexicalRichness(text)
    return lex.vocd(
        ntokens=ntokens,
        within_sample=within_sample,
        iterations=iterations
    )

def herdan(text):
    lex = LexicalRichness(text)
    return lex.Herdan

def summer(text):
    lex = LexicalRichness(text)
    return lex.Summer

def dugast(text):
    lex = LexicalRichness(text)
    return lex.Dugast

def maas(text):
    lex = LexicalRichness(text)
    return lex.Maas

def yulek(text):
    lex = LexicalRichness(text)
    return lex.yulek

def herdan_vm(text):
    lex = LexicalRichness(text)
    return lex.herdanvm

def simpson_d(text):
    lex = LexicalRichness(text)
    return lex.simpsond


# ============================================================
# 2. SAFE WRAPPER
# ============================================================

def safe_lexical_measure(text, func):
    """
    Computes one lexical measure safely.
    Returns NaN if the text is empty or if the metric cannot be computed.
    """
    if pd.isna(text):
        return np.nan

    text = str(text).strip()

    if text == "":
        return np.nan

    try:
        value = func(text)

        if value is None:
            return np.nan

        if isinstance(value, (int, float, np.number)):
            if np.isinf(value):
                return np.nan

        return value

    except Exception:
        return np.nan


def count_lexical_measures(text):
    """
    Computes lexical richness measures for one participant-level text.
    """

    result = {
        "word_count": safe_lexical_measure(text, word_count),
        "term_count": safe_lexical_measure(text, term_count),
        "ttr": safe_lexical_measure(text, ttr),
        "mtld": safe_lexical_measure(text, mtld),
        "hdd": safe_lexical_measure(text, hdd),
        "vocd": safe_lexical_measure(text, vocd),
        "herdan": safe_lexical_measure(text, herdan),
        "summer": safe_lexical_measure(text, summer),
        "dugast": safe_lexical_measure(text, dugast),
        "maas": safe_lexical_measure(text, maas),
        "yulek": safe_lexical_measure(text, yulek),
        "herdan_vm": safe_lexical_measure(text, herdan_vm),
        "simpson_d": safe_lexical_measure(text, simpson_d),
    }

    return pd.Series(result)


# ============================================================
# 3. LEXICAL CORRELATION ANALYSIS
# ============================================================

def lexical_correlation_analysis(
    df,
    speaker_filter=["Participant"],
    text_col="value",
    participant_col="Participant_ID",
    speaker_col="speaker",
    phq_col="PHQ_Score_x"
):
    """
    Performs lexical richness analysis.

    Steps:
    1. Optionally filters speakers, e.g. only Participant utterances.
    2. Concatenates all selected utterances per participant.
    3. Computes lexical richness measures per participant.
    4. Computes Pearson and Spearman correlations with PHQ score.
    """

    data = df.copy()

    # Optional: nur bestimmte Speaker behalten
    if speaker_filter is not None:
        speaker_filter_lower = [s.lower() for s in speaker_filter]

        data = data[
            data[speaker_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(speaker_filter_lower)
        ].copy()

    # Leere Texte vermeiden
    data[text_col] = data[text_col].fillna("").astype(str)

    # Text pro Teilnehmer:in zusammenführen
    participant_df = (
        data
        .groupby(participant_col)
        .agg({
            phq_col: "first",
            text_col: lambda x: " ".join(x.astype(str))
        })
        .reset_index()
        .rename(columns={text_col: "text"})
    )

    # Anzahl der verwendeten Utterances ergänzen
    n_utterances = (
        data
        .groupby(participant_col)
        .size()
        .reset_index(name="n_utterances")
    )

    participant_df = participant_df.merge(
        n_utterances,
        on=participant_col,
        how="left"
    )

    # Lexical Measures berechnen
    lexical_counts = participant_df["text"].apply(count_lexical_measures)

    participant_df = pd.concat(
        [participant_df.reset_index(drop=True), lexical_counts.reset_index(drop=True)],
        axis=1
    )

    # Korrelationen berechnen
    metric_cols = lexical_counts.columns.tolist()

    results = []

    for metric in metric_cols:
        temp = (
            participant_df[[participant_col, phq_col, metric]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        # Korrelation ist nicht sinnvoll, wenn PHQ oder Metrik konstant ist
        if temp[metric].nunique() <= 1 or temp[phq_col].nunique() <= 1:
            continue

        pearson_r, pearson_p = pearsonr(temp[metric], temp[phq_col])
        spearman_r, spearman_p = spearmanr(temp[metric], temp[phq_col])

        results.append({
            "metric": metric,
            "n": len(temp),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p
        })

    corr_df = pd.DataFrame(results)

    if len(corr_df) > 0:
        corr_df = corr_df.sort_values("pearson_p")

    return participant_df, corr_df, data


# ============================================================
# 4. DATA LOAD
# ============================================================

df = pd.read_csv("/home/jovyan/thesisnew-datavol-1/input/DAIC_FULL_NO_TAGS.csv")


# ============================================================
# 5. ONLY PARTICIPANT UTTERANCES
# ============================================================

participant_lexical, corr_lexical, df_lexical_filtered = lexical_correlation_analysis(
    df,
    speaker_filter=["Participant"],
    phq_col="PHQ_Score_x"
)

corr_lexical
# ============================================================
# 6. INCLUDING THERAPIST UTTERANCES
# ============================================================

participant_lexical_all, corr_lexical_all, df_lexical_filtered_all = lexical_correlation_analysis(
    df,
    speaker_filter=["Participant", "Ellie"],
    phq_col="PHQ_Score_x"
)

corr_lexical_all
# ============================================================
# 7. PLOT FUNCTION
# ============================================================

def plot_lexical_correlations(
    corr_df,
    title,
    alpha=0.05,
    figsize=(8.5, 5.2),
    dpi=200
):
    """
    Plots Pearson correlations with PHQ score.
    Significant correlations are highlighted using raw p-values.
    """

    correlation_df = corr_df.sort_values("pearson_r").copy()

    if correlation_df.empty:
        print("No correlations to plot.")
        return

    correlation_df["significant"] = correlation_df["pearson_p"] < alpha

    fig_height = max(figsize[1], 0.42 * len(correlation_df))
    fig, ax = plt.subplots(figsize=(figsize[0], fig_height), dpi=dpi)

    colors = correlation_df["significant"].map({
        True: "#4C78A8",
        False: "#CFCFCF"
    })

    bars = ax.barh(
        correlation_df["metric"],
        correlation_df["pearson_r"],
        color=colors,
        edgecolor="#555555",
        linewidth=0.8
    )

    # Nicht-signifikante Balken schraffieren
    for bar, is_sig in zip(bars, correlation_df["significant"]):
        if not is_sig:
            bar.set_hatch("//")

    # Nulllinie
    ax.axvline(
        0,
        color="#333333",
        linestyle="--",
        linewidth=1
    )

    # ============================================================
    # ADD SIGNIFICANCE LABELS
    # ============================================================

    max_abs = np.nanmax(np.abs(correlation_df["pearson_r"]))

    if np.isnan(max_abs) or max_abs == 0:
        max_abs = 0.1

    offset = max_abs * 0.035

    for bar, r, p, is_sig in zip(
        bars,
        correlation_df["pearson_r"],
        correlation_df["pearson_p"],
        correlation_df["significant"]
    ):
        if pd.isna(p):
            label = "n/a"
        elif is_sig:
            label = f"p = {p:.3f}"
        else:
            label = f"n.s. | p = {p:.3f}"

        x = bar.get_width()
        y = bar.get_y() + bar.get_height() / 2

        ax.text(
            x + offset if x >= 0 else x - offset,
            y,
            label,
            va="center",
            ha="left" if x >= 0 else "right",
            fontsize=8.5
        )

    # ============================================================
    # STYLING
    # ============================================================

    ax.set_xlabel("Pearson correlation with PHQ score", fontsize=12)
    ax.set_ylabel("Lexical metric", fontsize=12)

    ax.set_title(
        title,
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    ax.set_xlim(-max_abs * 1.65, max_abs * 1.65)

    ax.grid(axis="x", alpha=0.25, linewidth=0.8)
    ax.set_axisbelow(True)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.spines["left"].set_alpha(0.4)
    ax.spines["bottom"].set_alpha(0.4)

    ax.tick_params(axis="both", labelsize=10)

    legend_elements = [
        Patch(
            facecolor="#4C78A8",
            edgecolor="#555555",
            label="Significant, p < .05"
        ),
        Patch(
            facecolor="#CFCFCF",
            edgecolor="#555555",
            hatch="//",
            label="Not significant, p ≥ .05"
        )
    ]

    ax.legend(
        handles=legend_elements,
        frameon=False,
        loc="lower right",
        fontsize=9
    )

    plt.tight_layout()
    plt.show()
# ============================================================
# 8. PLOT ONLY PARTICIPANT UTTERANCES
# ============================================================

plot_lexical_correlations(
    corr_lexical,
    title="Correlation per Lexical Metric to PHQ Score\nOnly Participant Utterances"
)
# ============================================================
# 9. PLOT INCLUDING THERAPIST UTTERANCES
# ============================================================

plot_lexical_correlations(
    corr_lexical_all,
    title="Correlation per Lexical Metric to PHQ Score\nIncluding Therapist Utterances"
)